# Задача на 6-ом шаге урока

**⚖️ Взвешенный блендинг**

Задание: У тебя есть информация о точности модели на лидерборде (accuracy_score). Рассчитай веса, с которыми нужно будет взвесить предсказания моделей, чтобы они были пропорциональны accuracy_score, а их сумма равнялась единице.

Примечание: Ввод и вывод уже написан за вас.

In [3]:
n = int(input())
s = input().split()
scores = []
total = 0

for i in range(n):
    val = float(s[i])
    scores.append(val)
    total += val

weights = []

for i in range(n):
    weight = scores[i] / total
    weights.append(round(weight, 7))

print(*weights)

3
0.3 0.5 0.4
0.25 0.4166667 0.3333333


Sample Input:

3

0.3 0.5 0.4

Sample Output:

0.25 0.4166667 0.3333333

# Задача на 7-ом шаге урока

**🏄‍♂️ Hard voting**

Датасет с прогнозами моделей тут:

In [1]:
path = 'https://stepik.org/media/attachments/lesson/825510/make_soft_blend.csv'

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(path)

In [9]:
model_cols = [f'model_{i}' for i in range(1, 6)]


df['target_class'] = df[model_cols].mode(axis=1)[0].values

result = df[['car_id', 'target_class']]
result.to_csv('results/submission.csv', index=False)
result

,car_id,target_class
0,car_0,electro_bug
1,car_1,engine_overheat
2,car_2,engine_overheat
3,car_3,electro_bug
4,car_4,engine_check
...,...,...
1908,car_1908,break_bug
1909,car_1909,break_bug
1910,car_1910,engine_overheat
1911,car_1911,engine_check


Файл с решением должен выглядет так с заполненными значениями target_class

In [10]:
path = 'https://stepik.org/media/attachments/lesson/825510/sb_sample_submission.csv'

In [11]:
pd.read_csv(path).head()

,car_id,target_class
0,car_0,NaN
1,car_1,NaN
2,car_2,NaN
3,car_3,NaN
4,car_4,NaN


# Задача на 8-ом шаге урока

**🤝Soft Voting**

Датасета с предсказаниями поломок машин от разных моделей:

In [66]:
path_catboost_preds = 'https://stepik.org/media/attachments/lesson/825510/catboost_preds.csv'
path_lgbm_preds = 'https://stepik.org/media/attachments/lesson/825510/lgbm_preds.csv'
path_xgb_preds = 'https://stepik.org/media/attachments/lesson/825510/xgb_preds.csv'

catboost_preds_df = pd.read_csv(path_catboost_preds).reset_index().rename(columns={'index': 'car_id'})
lgbm_preds_df = pd.read_csv(path_lgbm_preds).reset_index().rename(columns={'index': 'car_id'})
xgb_preds_df = pd.read_csv(path_xgb_preds).reset_index().rename(columns={'index': 'car_id'})

In [76]:
result_df = catboost_preds_df + lgbm_preds_df + xgb_preds_df
result_df['soft_preds'] = (
    result_df
    .set_index('car_id')
    .rename(columns=lambda x: x.replace('_proba', ''))
    .idxmax(axis=1)
    .reset_index(drop=True)
)

result_df['soft_preds'].to_csv('results/soft_blend_submission.csv')
result_df.head()


,car_id,another_bug_proba,break_bug_proba,electro_bug_proba,engine_check_proba,engine_fuel_proba,engine_ignition_proba,engine_overheat_proba,gear_stick_proba,wheel_shake_proba,soft_preds
0,0,0.337219,0.344227,0.353973,0.314100,0.346557,0.322595,0.336272,0.352935,0.292123,electro_bug
1,3,0.479751,0.442886,0.292707,0.333411,0.280346,0.284728,0.328361,0.325581,0.232230,another_bug
2,6,0.348240,0.342305,0.317633,0.329424,0.307271,0.362316,0.348468,0.359000,0.285343,engine_ignition
3,9,0.339210,0.347235,0.317740,0.374427,0.301501,0.341304,0.311298,0.377495,0.289790,gear_stick
4,12,0.359307,0.372359,0.335765,0.314143,0.331229,0.346983,0.335586,0.364043,0.240584,break_bug


Ваша задача  — сблендить предсказания этих трех моделей при помощи методики мягкого голосования (soft voting). В качестве ответа отправьте файл, в котором для каждой строчки будет предсказанный ответ, записанный в колонку soft_preds. Названия ответа можно брать из названия соответствующей колонки, обрезая _probа. Порядок строк в высылаемом решении должен быть таким же, как и в файлах с предсказаниями моделей.



# Задача на 10-ом шаге урока

**Напишите свой Rank Averaging**

Реализуйте процедуру нормализации вероятностей, как описывалось в предыдущем шаге (https://stepik.org/lesson/825510/step/9?unit=829017). Для создания тестового датасета можно воспользоваться кодом:

In [77]:
import numpy as np
import pandas as pd
np.random.seed(42)
proba = np.random.random(100)
df = pd.DataFrame({'id': np.arange(len(proba)), 'proba': proba})
df

,id,proba
0,0,0.374540
1,1,0.950714
2,2,0.731994
3,3,0.598658
4,4,0.156019
...,...,...
95,95,0.493796
96,96,0.522733
97,97,0.427541
98,98,0.025419


Вам нужно посчитать отнормированные предсказания по инструкции из предыдущего степа. Финальный ответ запишите в столбец normalized_proba

In [81]:
import numpy as np
import pandas as pd
df['normalized_proba'] = df['proba'].rank() / df.shape[0]

# Задача на 11-ом шаге урока

**Rank Averaging для 3-х моделей**

В этот раз вам даны вероятности сразу для трех моделей -  proba1, proba2, proba3. Для создания тестового датасета можно использовать код:




In [82]:
import numpy as np
import pandas as pd
np.random.seed(42)
df = pd.DataFrame({'id': np.arange(100),
                   'proba1': np.random.random(100),
                   'proba2': np.random.random(100),
                   'proba3': np.random.random(100)})

Посчитайте для каждой модели ее ранг (как в прошлом степе) и потом запишите среднее значение этих трех рангов в колонку mean_rank.

In [88]:
import numpy as np
import pandas as pd

cols = ['proba1', 'proba2', 'proba3']
df['mean_rank'] = (df[cols].rank() / df.shape[0]).mean(axis=1)